# Lab 10: Automated Multi-Format Sales Report Pipeline and Delivery

**Objective:** Design and implement a production-ready automated report pipeline that extracts, transforms, and generates sales reports in multiple formats (Excel, HTML, PDF) and optionally delivers them via email.

**Reference:** Note 6.6 – Code Implementation

---
## Step 0: Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import os
import smtplib
import matplotlib
matplotlib.use('Agg') 
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from datetime import datetime
from email.mime.multipart import MIMEMultipart
from email.mime.base import MIMEBase
from email.mime.text import MIMEText
from email import encoders

# Excel
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# Jinja2 for HTML
from jinja2 import Template

# FPDF2 for PDF
from fpdf import FPDF

print("All libraries imported successfully!")
print(f"pandas version  : {pd.__version__}")
print(f"openpyxl version: {openpyxl.__version__}")

All libraries imported successfully!
pandas version  : 3.0.0
openpyxl version: 3.1.5


---
## Step 1: EXTRACT – Load Sales Data

In the **Extract** phase, raw data is loaded from its source. The goal is simply to bring the data into memory with no modifications yet.

In [ ]:
np.random.seed(42)

months   = ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]
products = ["Laptop", "Tablet", "Headphones", "Monitor", "Keyboard"]
regions  = ["North", "South", "East", "West"]

# Fixed unit prices per product
unit_prices = {
    "Laptop"     : 55000,
    "Tablet"     : 35000,
    "Headphones" :  3000,
    "Monitor"    : 15000,
    "Keyboard"   :  2000,
}

rows = []
for month in months:
    for product in products:
        for region in regions:
            units = np.random.randint(5, 50)
            rows.append({
                "Month"      : month,
                "Product"    : product,
                "Units_Sold" : units,
                "Unit_Price" : unit_prices[product],
                "Region"     : region,
            })

df_raw = pd.DataFrame(rows)

print(f"Raw dataset shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
df_raw.head(10)

Raw dataset shape: (120, 5)
Columns: ['Month', 'Product', 'Units_Sold', 'Unit_Price', 'Region']


,Month,Product,Units_Sold,Unit_Price,Region
0,Jan,Laptop,43,55000,North
1,Jan,Laptop,33,55000,South
2,Jan,Laptop,19,55000,East
3,Jan,Laptop,47,55000,West
4,Jan,Tablet,12,35000,North
5,Jan,Tablet,25,35000,South
6,Jan,Tablet,43,35000,East
7,Jan,Tablet,23,35000,West
8,Jan,Headphones,27,3000,North
9,Jan,Headphones,15,3000,South


---
## Step 2: TRANSFORM – Clean and Enrich Data

In the **Transform** phase we:
- Calculate `Revenue = Units_Sold × Unit_Price`
- Aggregate to product-level summaries (Total Units, Total Revenue, Average Price, Revenue Share %)
- Aggregate to monthly totals for charting

In [3]:
# ── 2a. Calculate Revenue ──────────────────────────────────────────────────
df_raw["Revenue"] = df_raw["Units_Sold"] * df_raw["Unit_Price"]

print("Sample rows after Revenue calculation:")
df_raw.head(6)

Sample rows after Revenue calculation:


,Month,Product,Units_Sold,Unit_Price,Region,Revenue
0,Jan,Laptop,43,55000,North,2365000
1,Jan,Laptop,33,55000,South,1815000
2,Jan,Laptop,19,55000,East,1045000
3,Jan,Laptop,47,55000,West,2585000
4,Jan,Tablet,12,35000,North,420000
5,Jan,Tablet,25,35000,South,875000


In [4]:
# ── 2b. Product-level summary ──────────────────────────────────────────────
product_summary = (
    df_raw
    .groupby("Product", as_index=False)
    .agg(
        Total_Units   = ("Units_Sold", "sum"),
        Total_Revenue = ("Revenue",    "sum"),
        Avg_Price     = ("Unit_Price", "mean"),
    )
)

total_rev = product_summary["Total_Revenue"].sum()
product_summary["Revenue_Share_Pct"] = (
    product_summary["Total_Revenue"] / total_rev * 100
).round(2)

product_summary = product_summary.sort_values("Total_Revenue", ascending=False).reset_index(drop=True)

print("Product Summary:")
product_summary

Product Summary:


,Product,Total_Units,Total_Revenue,Avg_Price,Revenue_Share_Pct
0,Laptop,684,37620000,55000.0,50.17
1,Tablet,724,25340000,35000.0,33.80
2,Monitor,591,8865000,15000.0,11.82
3,Headphones,607,1821000,3000.0,2.43
4,Keyboard,666,1332000,2000.0,1.78


In [5]:
# ── 2c. Monthly total revenue ──────────────────────────────────────────────
month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]

monthly_revenue = (
    df_raw
    .groupby("Month", as_index=False)
    .agg(Monthly_Revenue = ("Revenue", "sum"))
)
monthly_revenue["Month"] = pd.Categorical(
    monthly_revenue["Month"], categories=month_order, ordered=True
)
monthly_revenue = monthly_revenue.sort_values("Month").reset_index(drop=True)

print("Monthly Revenue:")
monthly_revenue

Monthly Revenue:


,Month,Monthly_Revenue
0,Jan,13671000
1,Feb,12334000
2,Mar,9846000
3,Apr,11902000
4,May,12949000
5,Jun,14276000


---
## Step 3: Prepare the `reports/` Output Folder

In [6]:
# Create reports directory if it does not exist
REPORTS_DIR = "reports"
os.makedirs(REPORTS_DIR, exist_ok=True)

# Date string for filenames
date_str = datetime.now().strftime("%Y%m%d")

EXCEL_PATH = os.path.join(REPORTS_DIR, f"sales_report_{date_str}.xlsx")
HTML_PATH  = os.path.join(REPORTS_DIR, f"sales_report_{date_str}.html")
PDF_PATH   = os.path.join(REPORTS_DIR, f"sales_report_{date_str}.pdf")
CHART_PATH = os.path.join(REPORTS_DIR, f"monthly_revenue_chart_{date_str}.png")

print(f"Reports will be saved to: ./{REPORTS_DIR}/")
print(f"  Excel : {EXCEL_PATH}")
print(f"  HTML  : {HTML_PATH}")
print(f"  PDF   : {PDF_PATH}")

Reports will be saved to: ./reports/
  Excel : reports\sales_report_20260312.xlsx
  HTML  : reports\sales_report_20260312.html
  PDF   : reports\sales_report_20260312.pdf


---
## Step 4: GENERATE – Excel Report (openpyxl)

We generate an Excel workbook with two sheets:
- **Product Summary** – styled headers, formatted currency columns
- **Monthly Revenue** – data used for the bar chart

In [ ]:
def generate_excel(product_summary: pd.DataFrame,
                   monthly_revenue: pd.DataFrame,
                   path: str) -> None:
    """Generate a styled Excel report with two sheets."""

    wb = openpyxl.Workbook()

    # ── Helper styles ──────────────────────────────────────────────────────
    header_fill  = PatternFill("solid", fgColor="1F4E79")   
    alt_fill     = PatternFill("solid", fgColor="D6E4F0")   
    header_font  = Font(color="FFFFFF", bold=True, size=11)
    body_font    = Font(size=11)
    center_align = Alignment(horizontal="center", vertical="center")
    thin = Side(style="thin", color="BFBFBF")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    # ── Sheet 1: Product Summary ───────────────────────────────────────────
    ws1 = wb.active
    ws1.title = "Product Summary"

    headers = ["Product", "Total Units", "Total Revenue (Rs)",
               "Avg Unit Price (Rs)", "Revenue Share (%)"]
    ws1.append(headers)

    # Style header row
    for col_idx, _ in enumerate(headers, start=1):
        cell = ws1.cell(row=1, column=col_idx)
        cell.fill      = header_fill
        cell.font      = header_font
        cell.alignment = center_align
        cell.border    = border

    # Data rows
    for row_idx, row in product_summary.iterrows():
        data_row = [
            row["Product"],
            int(row["Total_Units"]),
            float(row["Total_Revenue"]),
            float(row["Avg_Price"]),
            float(row["Revenue_Share_Pct"]),
        ]
        ws1.append(data_row)
        excel_row = row_idx + 2   # +1 for header, +1 for 1-indexing
        fill = alt_fill if row_idx % 2 == 0 else PatternFill()
        for col_idx in range(1, len(headers) + 1):
            cell = ws1.cell(row=excel_row, column=col_idx)
            cell.fill      = fill
            cell.font      = body_font
            cell.alignment = center_align
            cell.border    = border
            # Format currency columns
            if col_idx in (3, 4):
                cell.number_format = '#,##0.00'
            elif col_idx == 5:
                cell.number_format = '0.00"%"'

    # Auto-fit column widths
    col_widths = [18, 14, 22, 22, 18]
    for i, width in enumerate(col_widths, start=1):
        ws1.column_dimensions[get_column_letter(i)].width = width

    # ── Sheet 2: Monthly Revenue ───────────────────────────────────────────
    ws2 = wb.create_sheet(title="Monthly Revenue")
    ws2.append(["Month", "Total Revenue (Rs)"])
    for col_idx in range(1, 3):
        cell = ws2.cell(row=1, column=col_idx)
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = center_align

    for _, row in monthly_revenue.iterrows():
        ws2.append([str(row["Month"]), float(row["Monthly_Revenue"])])

    ws2.column_dimensions["A"].width = 12
    ws2.column_dimensions["B"].width = 22

    wb.save(path)
    print(f"[Excel] Saved → {path}")


generate_excel(product_summary, monthly_revenue, EXCEL_PATH)

[Excel] Saved → reports\sales_report_20260312.xlsx


---
## Step 5: GENERATE – HTML Report (Jinja2)

We use a **Jinja2 template** to render a fully styled HTML report that includes the product summary table and the total revenue figure.

In [8]:
HTML_TEMPLATE = """
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>Sales Report – {{ report_date }}</title>
  <style>
    body  { font-family: Arial, sans-serif; margin: 40px; color: #333; }
    h1    { color: #1F4E79; border-bottom: 3px solid #1F4E79; padding-bottom: 8px; }
    h2    { color: #2E75B6; margin-top: 30px; }
    .meta { color: #666; font-size: 14px; margin-bottom: 20px; }
    table { border-collapse: collapse; width: 100%; margin-top: 12px; }
    th    { background: #1F4E79; color: #fff; padding: 10px 14px; text-align: center; }
    td    { padding: 9px 14px; text-align: center; border: 1px solid #ddd; }
    tr:nth-child(even) td { background: #D6E4F0; }
    tr:hover td { background: #BDD7EE; }
    .total-box {
      margin-top: 20px; padding: 14px 20px;
      background: #1F4E79; color: #fff;
      display: inline-block; border-radius: 6px; font-size: 16px;
    }
    footer { margin-top: 40px; font-size: 12px; color: #aaa; }
  </style>
</head>
<body>
  <h1>📊 Automated Sales Report</h1>
  <p class="meta">Generated on: <strong>{{ report_date }}</strong></p>

  <h2>Product Summary</h2>
  <table>
    <thead>
      <tr>
        <th>#</th>
        <th>Product</th>
        <th>Total Units Sold</th>
        <th>Total Revenue (Rs)</th>
        <th>Avg Unit Price (Rs)</th>
        <th>Revenue Share (%)</th>
      </tr>
    </thead>
    <tbody>
      {% for row in rows %}
      <tr>
        <td>{{ loop.index }}</td>
        <td>{{ row.Product }}</td>
        <td>{{ "{:,}".format(row.Total_Units) }}</td>
        <td>{{ "{:,.2f}".format(row.Total_Revenue) }}</td>
        <td>{{ "{:,.2f}".format(row.Avg_Price) }}</td>
        <td>{{ "{:.2f}".format(row.Revenue_Share_Pct) }}%</td>
      </tr>
      {% endfor %}
    </tbody>
  </table>

  <div class="total-box">
    💰 Grand Total Revenue: Rs {{ "{:,.2f}".format(total_revenue) }}
  </div>

  <h2>Monthly Revenue Summary</h2>
  <table>
    <thead>
      <tr><th>Month</th><th>Revenue (Rs)</th></tr>
    </thead>
    <tbody>
      {% for m in monthly %}
      <tr>
        <td>{{ m.Month }}</td>
        <td>{{ "{:,.2f}".format(m.Monthly_Revenue) }}</td>
      </tr>
      {% endfor %}
    </tbody>
  </table>

  <footer>Auto-generated by Sales Report Pipeline &copy; {{ year }}</footer>
</body>
</html>
"""


def generate_html(product_summary: pd.DataFrame,
                  monthly_revenue: pd.DataFrame,
                  path: str) -> None:
    """Render and save an HTML report using a Jinja2 template."""

    template = Template(HTML_TEMPLATE)

    html_content = template.render(
        report_date   = datetime.now().strftime("%d %B %Y, %H:%M"),
        year          = datetime.now().year,
        rows          = product_summary.to_dict(orient="records"),
        monthly       = monthly_revenue.to_dict(orient="records"),
        total_revenue = product_summary["Total_Revenue"].sum(),
    )

    with open(path, "w", encoding="utf-8") as f:
        f.write(html_content)

    print(f"[HTML] Saved → {path}")


generate_html(product_summary, monthly_revenue, HTML_PATH)

[HTML] Saved → reports\sales_report_20260312.html


---
## Step 6: GENERATE – Bar Chart of Monthly Revenue (Matplotlib)

We create a bar chart that will be **embedded inside the PDF report**.

In [9]:
def generate_bar_chart(monthly_revenue: pd.DataFrame, chart_path: str) -> None:
    """Save a bar chart of monthly revenue as a PNG image."""

    fig, ax = plt.subplots(figsize=(9, 4.5))

    months = monthly_revenue["Month"].astype(str).tolist()
    values = monthly_revenue["Monthly_Revenue"].tolist()
    colors = plt.cm.Blues(np.linspace(0.45, 0.9, len(months)))

    bars = ax.bar(months, values, color=colors, edgecolor="#1F4E79", linewidth=0.8)

    # Value labels on top of each bar
    for bar, val in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + max(values) * 0.01,
            f"Rs {val/1e6:.1f}M",
            ha="center", va="bottom", fontsize=9, color="#1F4E79", fontweight="bold"
        )

    ax.set_title("Monthly Revenue Overview", fontsize=14, fontweight="bold",
                 color="#1F4E79", pad=14)
    ax.set_xlabel("Month", fontsize=11)
    ax.set_ylabel("Revenue (Rs)", fontsize=11)
    ax.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda x, _: f"Rs {x/1e6:.1f}M")
    )
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

    plt.tight_layout()
    plt.savefig(chart_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"[Chart] Saved → {chart_path}")


generate_bar_chart(monthly_revenue, CHART_PATH)

[Chart] Saved → reports\monthly_revenue_chart_20260312.png


C:\Users\shahi\AppData\Local\Temp\ipykernel_7428\94432565.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Step 7: GENERATE – PDF Report (FPDF2)

The PDF includes:
- A styled title page header
- Product summary table with alternating row colors
- The bar chart of monthly revenue embedded as an image

In [10]:
def generate_pdf(product_summary: pd.DataFrame,
                 monthly_revenue: pd.DataFrame,
                 chart_path: str,
                 path: str) -> None:
    """Generate a styled PDF report with a table and embedded bar chart."""

    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()

    # ── Header banner ──────────────────────────────────────────────────────
    pdf.set_fill_color(31, 78, 121)          # dark blue
    pdf.rect(0, 0, 210, 28, style="F")
    pdf.set_text_color(255, 255, 255)
    pdf.set_font("Helvetica", "B", 18)
    pdf.set_xy(0, 7)
    pdf.cell(210, 10, "Automated Sales Report", align="C")

    pdf.set_font("Helvetica", "", 10)
    pdf.set_xy(0, 19)
    pdf.cell(210, 6,
             f"Generated: {datetime.now().strftime('%d %B %Y, %H:%M')}",
             align="C")

    # ── Section: Product Summary ───────────────────────────────────────────
    pdf.set_text_color(31, 78, 121)
    pdf.set_font("Helvetica", "B", 13)
    pdf.set_xy(10, 34)
    pdf.cell(0, 8, "Product Summary", ln=True)

    # Table header
    col_widths = [38, 28, 40, 40, 34]
    headers    = ["Product", "Units Sold", "Revenue (Rs)",
                  "Avg Price (Rs)", "Share (%)"]

    pdf.set_fill_color(31, 78, 121)
    pdf.set_text_color(255, 255, 255)
    pdf.set_font("Helvetica", "B", 9)
    pdf.set_x(10)
    for header, width in zip(headers, col_widths):
        pdf.cell(width, 8, header, border=1, align="C", fill=True)
    pdf.ln()

    # Table rows with alternating colors
    pdf.set_font("Helvetica", "", 9)
    for i, row in product_summary.iterrows():
        if i % 2 == 0:
            pdf.set_fill_color(214, 228, 240)   # light blue
        else:
            pdf.set_fill_color(255, 255, 255)   # white
        pdf.set_text_color(50, 50, 50)
        pdf.set_x(10)

        row_data = [
            row["Product"],
            f"{int(row['Total_Units']):,}",
            f"{row['Total_Revenue']:,.2f}",
            f"{row['Avg_Price']:,.2f}",
            f"{row['Revenue_Share_Pct']:.2f}%",
        ]
        for val, width in zip(row_data, col_widths):
            pdf.cell(width, 7, str(val), border=1, align="C", fill=True)
        pdf.ln()

    # Total row
    pdf.set_fill_color(31, 78, 121)
    pdf.set_text_color(255, 255, 255)
    pdf.set_font("Helvetica", "B", 9)
    pdf.set_x(10)
    total_rev = product_summary["Total_Revenue"].sum()
    total_units = product_summary["Total_Units"].sum()
    totals = ["TOTAL", f"{int(total_units):,}",
              f"{total_rev:,.2f}", "-", "100.00%"]
    for val, width in zip(totals, col_widths):
        pdf.cell(width, 7, val, border=1, align="C", fill=True)
    pdf.ln(10)

    # ── Section: Monthly Revenue Bar Chart ────────────────────────────────
    pdf.set_text_color(31, 78, 121)
    pdf.set_font("Helvetica", "B", 13)
    pdf.cell(0, 8, "Monthly Revenue Bar Chart", ln=True)
    pdf.ln(2)

    if os.path.exists(chart_path):
        # Centre the chart on the page
        img_w, img_h = 170, 85
        x_pos = (210 - img_w) / 2
        pdf.image(chart_path, x=x_pos, y=pdf.get_y(), w=img_w, h=img_h)
        pdf.ln(img_h + 6)
    else:
        pdf.set_font("Helvetica", "I", 10)
        pdf.set_text_color(150, 0, 0)
        pdf.cell(0, 8, "[Chart image not found]", ln=True)

    # ── Footer ─────────────────────────────────────────────────────────────
    pdf.set_y(-14)
    pdf.set_font("Helvetica", "I", 8)
    pdf.set_text_color(170, 170, 170)
    pdf.cell(0, 6,
             f"Auto-generated by Sales Report Pipeline  |  Page {pdf.page_no()}",
             align="C")

    pdf.output(path)
    print(f"[PDF]   Saved → {path}")


generate_pdf(product_summary, monthly_revenue, CHART_PATH, PDF_PATH)

[PDF]   Saved → reports\sales_report_20260312.pdf


C:\Users\shahi\AppData\Local\Temp\ipykernel_7428\664579163.py:29: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 8, "Product Summary", ln=True)
C:\Users\shahi\AppData\Local\Temp\ipykernel_7428\664579163.py:81: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 8, "Monthly Revenue Bar Chart", ln=True)


---
## Step 8: DELIVER – Email Function (SMTP / Gmail)

The **Deliver** phase sends the generated report to a recipient via email using Python's built-in `smtplib` with **TLS encryption**.


In [ ]:
def send_report_email(
    sender_email   : str,
    app_password   : str,
    recipient_email: str,
    attachment_path: str,
    smtp_host      : str = "smtp.gmail.com",
    smtp_port      : int = 587,
) -> None:
    """
    Send a report file as an email attachment via SMTP (TLS).

    Parameters
    ----------
    sender_email    : Gmail address of the sender.
    app_password    : Gmail App Password (16-char string).
    recipient_email : Destination email address.
    attachment_path : Local path to the report file to attach.
    smtp_host       : SMTP server (default: Gmail).
    smtp_port       : SMTP port (587 = STARTTLS).
    """
    if not os.path.exists(attachment_path):
        print(f"[Email] ERROR: File not found – {attachment_path}")
        return

    # Build message
    msg = MIMEMultipart()
    msg["From"]    = sender_email
    msg["To"]      = recipient_email
    msg["Subject"] = f"Sales Report – {datetime.now().strftime('%d %B %Y')}"

    body = (
        "Hello,\n\n"
        "Please find attached the automated sales report generated today.\n\n"
        "This email was sent automatically by the Sales Report Pipeline.\n\n"
        "Regards,\nSales Analytics Team"
    )
    msg.attach(MIMEText(body, "plain"))

    # Attach the report file
    with open(attachment_path, "rb") as f:
        part = MIMEBase("application", "octet-stream")
        part.set_payload(f.read())

    encoders.encode_base64(part)
    part.add_header(
        "Content-Disposition",
        f'attachment; filename="{os.path.basename(attachment_path)}"',
    )
    msg.attach(part)

    # Send via SMTP with STARTTLS
    try:
        with smtplib.SMTP(smtp_host, smtp_port) as server:
            server.ehlo()
            server.starttls()          # Encrypt the connection
            server.ehlo()
            server.login(sender_email, app_password)
            server.sendmail(sender_email, recipient_email, msg.as_string())
        print(f"[Email] Report sent successfully to {recipient_email}")
    except smtplib.SMTPAuthenticationError:
        print("[Email] Authentication failed. Check your App Password.")
    except smtplib.SMTPException as e:
        print(f"[Email] SMTP error: {e}")


# ── USAGE () ─────────────────
# SENDER_EMAIL    = "youremail@gmail.com"
# APP_PASSWORD    = "xxxx xxxx xxxx xxxx"   # Gmail App Password
# RECIPIENT_EMAIL = "recipient@example.com"

# send_report_email(
#     sender_email    = SENDER_EMAIL,
#     app_password    = APP_PASSWORD,
#     recipient_email = RECIPIENT_EMAIL,
#     attachment_path = PDF_PATH,
# )

---
## Step 9: Verify Generated Reports

In [11]:
print("=" * 50)
print("  Report Generation Summary")
print("=" * 50)

for label, fpath in [("Excel", EXCEL_PATH), ("HTML", HTML_PATH), ("PDF", PDF_PATH)]:
    exists = os.path.exists(fpath)
    size   = os.path.getsize(fpath) / 1024 if exists else 0
    status = f"✅  {size:.1f} KB" if exists else "❌  NOT FOUND"
    print(f"  {label:<8}: {os.path.basename(fpath):<45} {status}")

print("=" * 50)
print(f"\nAll reports saved in: ./{REPORTS_DIR}/")

  Report Generation Summary
  Excel   : sales_report_20260312.xlsx                    ✅  5.9 KB
  HTML    : sales_report_20260312.html                    ✅  3.2 KB
  PDF     : sales_report_20260312.pdf                     ✅  48.6 KB

All reports saved in: ./reports/


---
## Step 10: Discussion Questions

### Q1. Explain the role of each step in the automated report pipeline: Extract → Transform → Generate → Deliver

| Phase | Role |
|-------|------|
| **Extract** | Loads raw data from source systems (CSV, database, API) into memory. No changes are made — data is read as-is. This ensures a clean separation between data retrieval and processing. |
| **Transform** | Cleans, enriches, and reshapes the data. Here we calculate `Revenue = Units_Sold × Unit_Price`, aggregate to product-level summaries, compute revenue share percentages, and build monthly totals. This is where business logic lives. |
| **Generate** | Converts the transformed data into human-readable output formats — Excel (structured workbook), HTML (web page via Jinja2), and PDF (printable document via FPDF2 with chart). Each format serves a different audience or use case. |
| **Deliver** | Distributes the finished reports to stakeholders automatically via email (SMTP/STARTTLS). This removes the need for manual file sharing and ensures timely delivery. |

---

### Q2. How does the pipeline ensure that Excel, HTML, and PDF reports are consistent in content?

All three reports are generated **from the same DataFrames** — `product_summary` and `monthly_revenue` — which are computed once in the Transform phase. Because every format function receives these same objects as arguments, the figures, totals, and row order are identical across all outputs. There is a single source of truth; changing the data once updates all formats simultaneously.

---

### Q3. Modify the pipeline to include a bar chart of monthly revenue in the PDF report. *(Implemented in Step 6 & 7 above)*

The modification involved two additions:
1. **`generate_bar_chart()`** – Uses `matplotlib` to draw a styled bar chart of monthly revenue and saves it as a PNG file inside the `reports/` folder.
2. **`generate_pdf()` updated** – After the product summary table, the function embeds the saved PNG into the PDF using `pdf.image()`, centering it on the page with appropriate dimensions.

---

### Q4. How would you secure email credentials when deploying this pipeline in production?

Hardcoding credentials in source code is a serious security risk. In production, the following strategies should be used:

1. **Environment Variables** – Store `SENDER_EMAIL` and `APP_PASSWORD` in OS environment variables and read them with `os.environ.get("APP_PASSWORD")`. This keeps secrets out of the codebase.
2. **`.env` file + `python-dotenv`** – Store credentials in a `.env` file, add it to `.gitignore`, and load it at runtime using the `python-dotenv` library.
3. **Secrets Managers** – On cloud platforms, use services such as AWS Secrets Manager, Azure Key Vault, or HashiCorp Vault to store and rotate credentials securely.
4. **CI/CD Secret Variables** – In automated pipelines (GitHub Actions, GitLab CI), inject secrets as masked environment variables so they never appear in logs.
5. **OAuth2 instead of App Passwords** – For Gmail in production, prefer OAuth2 authentication over static App Passwords, as tokens can be scoped and revoked without changing the underlying account password.

In [1]:
# Demonstration: secure credential loading using environment variables
import os


SENDER_EMAIL    = os.environ.get("REPORT_SENDER_EMAIL", "NOT_SET")
APP_PASSWORD    = os.environ.get("REPORT_APP_PASSWORD", "NOT_SET")
RECIPIENT_EMAIL = os.environ.get("REPORT_RECIPIENT",   "NOT_SET")

print(f"Sender email loaded from env: {'✅' if SENDER_EMAIL != 'NOT_SET' else '❌ Not set'}")
print(f"App password loaded from env: {'✅' if APP_PASSWORD != 'NOT_SET' else '❌ Not set'}")

Sender email loaded from env: ❌ Not set
App password loaded from env: ❌ Not set
